### Install Dependencies

In [0]:
!pip install databricks-sdk -U
dbutils.library.restartPython()

### Define Parameters

In [0]:
dbutils.widgets.text("GENIE_SPACE_ID", "", "Genie Space ID (from notebook 0)")
dbutils.widgets.text("QUESTION", "What are my top 5 campaigns by total sales?", "Question")
GENIE_SPACE_ID = dbutils.widgets.get("GENIE_SPACE_ID")
QUESTION = dbutils.widgets.get("QUESTION")


### Query Genie Space via API

Uses the [Genie Conversation API](https://docs.databricks.com/api/workspace/genie) to ask a natural-language question and poll for the answer.

In [0]:
import json, time
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Start conversation
resp = requests.post(
    f"{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/start-conversation",
    headers=headers,
    json={"content": QUESTION},
)
resp.raise_for_status()
msg = resp.json()
conversation_id = msg["conversation_id"]
message_id = msg["message_id"]
print(f"Conversation: {conversation_id}")

# Poll until complete
for _ in range(60):
    poll = requests.get(
        f"{host}/api/2.0/genie/spaces/{GENIE_SPACE_ID}/conversations/{conversation_id}/messages/{message_id}",
        headers=headers,
    )
    poll.raise_for_status()
    data = poll.json()
    status = data.get("status")
    if status in ("COMPLETED", "FAILED", "CANCELLED"):
        break
    time.sleep(2)

print("Status:", data.get("status"))
for att in data.get("attachments", []):
    if att.get("text"):
        print("\nAnswer:\n", att["text"][:2000])
